# RetailPulse: Feature Engineering

## Overview

Feature engineering is the process of creating meaningful analytical features from existing dataset columns. These features can help reveal business patterns, support more detailed analysis, and improve the quality of insights generated from the data.

In this notebook, the cleaned RetailPulse sales dataset will be transformed into an analysis-ready dataset by creating useful features from the existing order, date, quantity, amount, and categorical information.

## Objectives

* Load the cleaned sales dataset.
* Verify the dataset structure and data types.
* Create date-based features for time-series analysis.
* Develop sales-related features to support business analysis.
* Validate the newly created features.
* Save the feature-engineered dataset for exploratory data analysis and dashboard development.

## Approach

The notebook will follow a structured workflow, with each feature documented before its implementation. Validation checks will be performed to confirm that the new features are consistent with the underlying data.

The original cleaned dataset will remain unchanged. The feature-engineered dataset will be saved separately in the `data/processed/` directory.


## 1. Load the Cleaned Dataset

The cleaned dataset produced in the previous notebook will be loaded from the `data/processed/` directory.

The dataset's dimensions, column names, and data types will then be inspected to establish the starting point for feature engineering.
The date column will be converted to a datetime data type to support date-based feature creation. 


In [ ]:
import pandas as pd

cleaned_csv = "../data/processed/cleaned_dataset.csv"

df_cleaned = pd.read_csv(cleaned_csv)

print("Cleaned dataset loaded successfully.")
print(f"Dataset shape: {df_cleaned.shape}")
print(f"Number of rows: {df_cleaned.shape[0]}")
print(f"Number of columns: {df_cleaned.shape[1]}")

Cleaned dataset loaded successfully.
Dataset shape: (128969, 18)
Number of rows: 128969
Number of columns: 18


In [7]:
df_cleaned["date"] = pd.to_datetime(df_cleaned["date"])
print(f"Data type of date: {df_cleaned["date"].dtype}")

Data type of date: datetime64[us]


### 2. Inspect the Dataset Structure

Before creating new features, the dataset's existing columns, data types, and sample records will be inspected.

This step helps confirm that the expected fields are available and identifies the data types of the columns that will be used during feature engineering.


In [13]:
print(f"Column names: {df_cleaned.columns.to_list()}")

print(f"\nData types: \n{df_cleaned.dtypes}")

print("\nSample records:")
display(df_cleaned.head(10))

Column names: ['order_id', 'date', 'status', 'fulfilment', 'sales_channel', 'ship_service_level', 'style', 'sku', 'category', 'size', 'asin', 'courier_status', 'qty', 'amount', 'ship_state', 'ship_country', 'b2b', 'fulfilled_by']

Data types: 
order_id                         str
date                  datetime64[us]
status                           str
fulfilment                       str
sales_channel                    str
ship_service_level               str
style                            str
sku                              str
category                         str
size                             str
asin                             str
courier_status                   str
qty                            int64
amount                       float64
ship_state                       str
ship_country                     str
b2b                              str
fulfilled_by                     str
dtype: object

Sample records:


,order_id,date,status,fulfilment,sales_channel,ship_service_level,style,sku,category,size,asin,courier_status,qty,amount,ship_state,ship_country,b2b,fulfilled_by
0,405-8078784-5731545,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,SET389,SET389-KR-NP-S,Set,S,B09KXVBD7Z,Unshipped,0,647.62,MAHARASHTRA,IN,Customer,Easy Ship
1,171-9198151-1101146,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3781,JNE3781-KR-XXXL,kurta,3XL,B09K3WFS32,Shipped,1,406.00,KARNATAKA,IN,Customer,Easy Ship
2,404-0687676-7273146,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3371,JNE3371-KR-XL,kurta,XL,B07WV4JV4D,Shipped,1,329.00,MAHARASHTRA,IN,Business,Unknown
3,403-9615377-8133951,2022-04-30,Cancelled,Merchant,Amazon.in,Standard,J0341,J0341-DR-L,Western Dress,L,B099NRCT7B,Unshipped,0,753.33,PUDUCHERRY,IN,Customer,Easy Ship
4,407-1069790-7240320,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3671,JNE3671-TU-XXXL,Top,3XL,B098714BZP,Shipped,1,574.00,TAMIL NADU,IN,Customer,Unknown
5,404-1490984-4578765,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,SET264,SET264-KR-NP-XL,Set,XL,B08YN7XDSG,Shipped,1,824.00,UTTAR PRADESH,IN,Customer,Unknown
6,408-5748499-6859555,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,J0095,J0095-SET-L,Set,L,B08CMHNWBN,Shipped,1,653.00,CHANDIGARH,IN,Customer,Unknown
7,406-7807733-3785945,2022-04-30,Shipped - Delivered to Buyer,Merchant,Amazon.in,Standard,JNE3405,JNE3405-KR-S,kurta,S,B081WX4G4Q,Shipped,1,399.00,TELANGANA,IN,Customer,Easy Ship
8,407-5443024-5233168,2022-04-30,Cancelled,Amazon,Amazon.in,Expedited,SET200,SET200-KR-NP-A-XXXL,Set,3XL,B08L91ZZXN,Cancelled,0,0.00,TELANGANA,IN,Customer,Unknown
9,402-4393761-0311520,2022-04-30,Shipped,Amazon,Amazon.in,Expedited,JNE3461,JNE3461-KR-XXL,kurta,XXL,B08B3XF5MH,Shipped,1,363.00,TAMIL NADU,IN,Customer,Unknown


### 3. Verify Data Quality Before Feature Engineering

Before creating new analytical features, the dataset will be checked for missing values and duplicate records.

Although these checks were performed during data cleaning, repeating them after loading the processed CSV helps confirm that the saved dataset has been loaded correctly and is ready for feature engineering.


In [18]:
print(f"Total missing values: {df_cleaned.isnull().sum().sum()}")
print(f"Exact duplicate records: {df_cleaned.duplicated().sum()}")

print("\nMissing values by column:")
display(df_cleaned.isnull().sum().to_frame(name="missing_values"))

Total missing values: 0
Exact duplicate records: 0

Missing values by column:


,missing_values
order_id,0
date,0
status,0
fulfilment,0
sales_channel,0
ship_service_level,0
style,0
sku,0
category,0
size,0


### 4. Create Date-Based Features

Date-based features help identify temporal patterns in sales data. By extracting useful components from the existing `date` column, we can analyze sales performance across different time periods.

The following features will be created:

* **`year`** — The year of the order.
* **`month`** — The numerical month of the order.
* **`month_name`** — The full name of the month.
* **`day`** — The day of the month.

These features will support monthly trend analysis, weekday comparisons, and time-based visualizations in later stages of the project.


In [22]:
df_cleaned["year"] = df_cleaned["date"].dt.year
df_cleaned["month"] = df_cleaned["date"].dt.month
df_cleaned["month_name"] = df_cleaned["date"].dt.month_name()
df_cleaned["day"] = df_cleaned["date"].dt.day_name()

print("Date-based features created successfully.")
df_cleaned[["year", "month", "month_name", "day"]].sample(10)

Date-based features created successfully.


,year,month,month_name,day
97865,2022,6,June,Thursday
87201,2022,5,May,Monday
118766,2022,6,June,Tuesday
62145,2022,5,May,Saturday
52652,2022,5,May,Sunday
75269,2022,5,May,Tuesday
84688,2022,5,May,Wednesday
57101,2022,5,May,Wednesday
18904,2022,4,April,Tuesday
18044,2022,4,April,Tuesday


### 5. Inspect Sales and Quantity Columns

The `qty` and `amount` columns provide important information about order quantities and monetary values.

Before creating additional sales-related features, we will inspect their data types, descriptive statistics, and sample values. This will help determine which derived features are appropriate for the dataset and avoid duplicating information that already exists.


In [27]:
print("Column data types:")
display(df_cleaned[["qty", "amount"]].dtypes.to_frame(name="data_type"))

print("\nDescriptive statistics:")
display(df_cleaned[["qty", "amount"]].describe())

print("\nSample records:")
display(df_cleaned[["order_id", "qty", "amount"]].head(10))

Column data types:


,data_type
qty,int64
amount,float64



Descriptive statistics:


,qty,amount
count,128969.00000,128969.000000
mean,0.90445,609.371580
std,0.31333,313.335444
min,0.00000,0.000000
25%,1.00000,413.000000
50%,1.00000,583.000000
75%,1.00000,771.000000
max,15.00000,5584.000000



Sample records:


,order_id,qty,amount
0,405-8078784-5731545,0,647.62
1,171-9198151-1101146,1,406.00
2,404-0687676-7273146,1,329.00
3,403-9615377-8133951,0,753.33
4,407-1069790-7240320,1,574.00
5,404-1490984-4578765,1,824.00
6,408-5748499-6859555,1,653.00
7,406-7807733-3785945,1,399.00
8,407-5443024-5233168,0,0.00
9,402-4393761-0311520,1,363.00


### 6. Inspect Order Status Categories

The `status` column contains order-level status information that can help distinguish between different order outcomes.

Before deriving a broader order status category, the existing status values and their frequencies will be examined. This will help identify the categories present in the dataset and determine an appropriate grouping strategy.


In [29]:
print("Order status distribution:")
display(df_cleaned["status"].value_counts().to_frame(name="record_count"))

print("\nCourier status distribution:")
display(df_cleaned["courier_status"].value_counts().to_frame(name="record_count"))

Order status distribution:


,record_count
status,
Shipped,77801
Shipped - Delivered to Buyer,28769
Cancelled,18329
Shipped - Returned to Seller,1953
Shipped - Picked Up,973
Pending,658
Pending - Waiting for Pick Up,281
Shipped - Returning to Seller,145
Shipped - Out for Delivery,35



Courier status distribution:


,record_count
courier_status,
Shipped,109484
Unshipped,13553
Cancelled,5932


### 7. Create Order Status Groups

The `status` column contains several detailed order statuses. Grouping these statuses into broader categories can simplify order outcome analysis and improve dashboard readability.

A new `order_status_group` column will be created to classify records into Delivered, Cancelled, Returned, Rejected, In Progress, and Other categories.

The original `status` column will be retained so that the detailed order status remains available for more specific analysis.


In [30]:
status_mapping = {
    "Shipped - Delivered to Buyer": "Delivered",
    "Cancelled": "Cancelled",
    "Shipped - Returned to Seller": "Returned",
    "Shipped - Returning to Seller": "Returned",
    "Shipped - Rejected by Buyer": "Rejected",
    "Shipped": "In Progress",
    "Shipped - Picked Up": "In Progress",
    "Shipped - Out for Delivery": "In Progress",
    "Pending": "In Progress",
    "Pending - Waiting for Pick Up": "In Progress",
    "Shipping": "In Progress",
    "Shipped - Lost in Transit": "Other",
    "Shipped - Damaged": "Other"
}

df_cleaned["order_status_group"] = df_cleaned["status"].map(status_mapping)

print("Order status groups created successfully.")

display(
    df_cleaned["order_status_group"]
    .value_counts(dropna=False)
    .to_frame(name="record_count")
)

Order status groups created successfully.


,record_count
order_status_group,
In Progress,79756
Delivered,28769
Cancelled,18329
Returned,2098
Rejected,11
Other,6


### 8. Inspect Product Size Categories

The `size` column identifies the size associated with each product. Examining its distinct values and frequency distribution will help determine whether creating broader size categories would add analytical value.

The existing size values will be inspected before any transformation is applied. The original column will be retained to preserve the detailed product information.


In [36]:
print(f"Number of distinct size values: {df_cleaned['size'].nunique()}")

print("\nProduct size distribution:")
df_cleaned["size"].value_counts()

Number of distinct size values: 11

Product size distribution:


size
M       22709
L       22130
XL      20875
XXL     18096
S       17090
3XL     14815
XS      11161
6XL       738
5XL       550
4XL       427
Free      378
Name: count, dtype: int64

### 9. Create Product Size Categories

The `size` column contains standard clothing sizes, extended sizes, and a Free size category. Grouping these values into broader categories will support comparisons between different size groups.

A new `size_category` column will classify each record as Standard, Extended, or Free Size. The original `size` column will be retained for detailed analysis.


In [37]:
size_mapping = {
    "XS": "Standard",
    "S": "Standard",
    "M": "Standard",
    "L": "Standard",
    "XL": "Standard",
    "XXL": "Standard",
    "3XL": "Extended",
    "4XL": "Extended",
    "5XL": "Extended",
    "6XL": "Extended",
    "Free": "Free Size"
}

df_cleaned["size_category"] = df_cleaned["size"].map(size_mapping)

print("Product size categories created successfully.")

df_cleaned["size_category"].value_counts()

Product size categories created successfully.


size_category
Standard     112061
Extended      16530
Free Size       378
Name: count, dtype: int64

### 10. Validate Engineered Features

The newly created features will be checked to confirm that their values have been assigned correctly and that no unexpected missing values have been introduced.

This validation covers the date-based features, order status groups, and product size categories. The checks will also confirm that the dataset retains its expected number of records.


In [48]:
engineered_columns = [
    "year",
    "month",
    "month_name",
    "day",
    "order_status_group",
    "size_category"
]

print("Unique values in engineered features:\n")
for column in engineered_columns:
    print(f"{column}: {df_cleaned[column].nunique()} unique values")

print("\nMissing values in engineered features:")
df_cleaned[engineered_columns].isnull().sum().to_frame(name="missing_values")

Unique values in engineered features:

year: 1 unique values
month: 4 unique values
month_name: 4 unique values
day: 7 unique values
order_status_group: 6 unique values
size_category: 3 unique values

Missing values in engineered features:


,missing_values
year,0
month,0
month_name,0
day,0
order_status_group,0
size_category,0


### 11. Create Revenue Feature

The `amount` column contains the recorded monetary value associated with each sales record. For this project, the `revenue` feature will include the recorded amount for orders classified as `Delivered` or `In Progress`.

Records belonging to the `Cancelled`, `Returned`, `Rejected`, and `Other` groups will be assigned a revenue value of zero.

This is a project-specific analytical definition based on order status. Since payment confirmation is not established by the available data, this metric represents the recorded value of eligible orders rather than verified accounting revenue.

In [55]:
df_cleaned["revenue"] = df_cleaned["amount"].where(
    df_cleaned["order_status_group"].isin(["Delivered", "In Progress"]),0)

print("Revenue feature created successfully.")
df_cleaned[["status", "order_status_group", "amount", "revenue"]].head(10)

Revenue feature created successfully.


,status,order_status_group,amount,revenue
0,Cancelled,Cancelled,647.62,0.0
1,Shipped - Delivered to Buyer,Delivered,406.00,406.0
2,Shipped,In Progress,329.00,329.0
3,Cancelled,Cancelled,753.33,0.0
4,Shipped,In Progress,574.00,574.0
5,Shipped,In Progress,824.00,824.0
6,Shipped,In Progress,653.00,653.0
7,Shipped - Delivered to Buyer,Delivered,399.00,399.0
8,Cancelled,Cancelled,0.00,0.0
9,Shipped,In Progress,363.00,363.0


In [64]:
print("Revenue validation:")

print(f"Total revenue amount: ₹{df_cleaned['amount'].sum():,.2f}")
print(f"Revenue from eligible orders: ₹{df_cleaned['revenue'].sum():,.2f}")

print("\nRevenue by order status group:")
df_cleaned.groupby("order_status_group")["revenue"].sum().sort_values(ascending=False)

Revenue validation:
Total revenue amount: ₹78,590,043.30
Revenue from eligible orders: ₹70,283,067.00

Revenue by order status group:


order_status_group
In Progress    51632252.0
Delivered      18650815.0
Cancelled             0.0
Other                 0.0
Rejected              0.0
Returned              0.0
Name: revenue, dtype: float64

### 12. Final Feature Engineering Validation

This section verifies the features created during the feature engineering process and checks that the dataset remains consistent.

The validation covers missing values in engineered columns, duplicate records, dataset dimensions, revenue calculations, and the distributions of the newly created categorical features.

These checks help confirm that the dataset is ready to be saved for exploratory data analysis and dashboard development.


In [73]:
engineered_columns = [
    "year",
    "month",
    "month_name",
    "day",
    "order_status_group",
    "size_category",
    "revenue"
]

print("Missing values in engineered features:")
display(
    df_cleaned[engineered_columns].isnull().sum().to_frame(name="missing_values")
)

expected_revenue = df_cleaned["amount"].where(
    df_cleaned["order_status_group"].isin(["Delivered", "In Progress"]),
    0
)

print(
    "\nRevenue calculation matches expected values:",
    df_cleaned["revenue"].equals(expected_revenue)
)

print("\nOrder status group distribution:")
display(df_cleaned["order_status_group"].value_counts().to_frame(name="record_count"))

print("\nProduct size category distribution:")
display(df_cleaned["size_category"].value_counts().to_frame(name="record_count"))

Missing values in engineered features:


,missing_values
year,0
month,0
month_name,0
day,0
order_status_group,0
size_category,0
revenue,0



Revenue calculation matches expected values: True

Order status group distribution:


,record_count
order_status_group,
In Progress,79756
Delivered,28769
Cancelled,18329
Returned,2098
Rejected,11
Other,6



Product size category distribution:


,record_count
size_category,
Standard,112061
Extended,16530
Free Size,378


### 13. Check Final Dataset Integrity

This section verifies the final dimensions of the feature-engineered dataset and checks for exact duplicate records.

The dataset is expected to retain all 128,969 records from the cleaned dataset, with additional columns created during feature engineering. These checks help confirm that the feature engineering process has not unexpectedly changed the record count or introduced duplicate rows.


In [74]:
print(f"Final dataset shape: {df_cleaned.shape}")
print(f"Total rows: {len(df_cleaned)}")
print(f"Total columns: {df_cleaned.shape[1]}")
print(f"Exact duplicate records: {df_cleaned.duplicated().sum()}")

Final dataset shape: (128969, 25)
Total rows: 128969
Total columns: 25
Exact duplicate records: 0


### 14. Save the Feature-Engineered Dataset

After completing feature engineering and validation, the updated dataset will be saved to the existing `cleaned_dataset.csv` file in the `data/processed/` directory.

This approach maintains a single processed dataset for subsequent analysis and Power BI development. The original raw dataset remains unchanged, and the cleaning and feature engineering steps can be reproduced by running their respective notebooks in sequence.

In [75]:
cleaned_csv = "../data/processed/cleaned_dataset.csv"

df_cleaned.to_csv(cleaned_csv, index=False)

print("Feature-engineered dataset saved successfully.")
print(f"File location: {cleaned_csv}")
print(f"Dataset shape: {df_cleaned.shape}")

Feature-engineered dataset saved successfully.
File location: ../data/processed/cleaned_dataset.csv
Dataset shape: (128969, 25)


## 15. Conclusion

The feature engineering process is complete. Seven new features have been created to support date-based analysis, order status analysis, product size comparisons, and sales value analysis.

The new `order_status_group` and `size_category` columns simplify categorical analysis, while the date-based features support time-based comparisons. The `revenue` column follows the project's defined business rule, including recorded amounts for Delivered and In Progress orders.

Validation confirmed that the engineered features contain no missing values, the revenue calculation follows the intended rule, and no exact duplicate records remain.

The final dataset contains **128,969 rows and 25 columns** and has been saved to `data/processed/cleaned_dataset.csv`, replacing the previous cleaned dataset.

The dataset is now ready for the next stage of RetailPulse: **Exploratory Data Analysis (EDA)**, where we will investigate sales patterns, product performance, order outcomes, and other business insights.
